# CompoundT5 on 147k ORD reactions — the clean-base analogue of variant 4

Second run of the ORD-free base line (see `12_train_reactant_compoundt5_57k.ipynb` for why
`sagawa/CompoundT5` and why the vocabulary repair is mandatory). Same recipe, 2.58x the data:
147,000 ORD reactions instead of 57,000.

**Why more data rather than more epochs.** The 57k run ended undertrained, not converged — its
`eval_loss` fell monotonically to the last step (1.719 -> 0.568, best checkpoint = the final one)
and the flattening over the third epoch tracks the linear scheduler annealing the learning rate
to ~0 (4.5e-4 at epoch 0.45, 1.5e-5 at epoch 2.92), not saturation. Extra epochs over the same
57,000 examples risk overfitting; unique reactions cannot, and a base that has never seen a
reaction is the case where new data should matter most.

**What this also settles.** On `ReactionT5v2-retrosynthesis` the same jump went the wrong way —
variant 4 (147k) scored 47.3% ORD top-1 against variant 2's (57k) 50.3%, recorded in `RESULTS.md`
as "more data does not help". That base had already been pretrained on 1.5M ORD reactions, so the
extra 90k carried little it had not seen. This run tests the same axis where that confound is
absent.

**Reference points on the 300-record ORD test:** untuned CompoundT5 0.0% exact / 14.7% valid;
CompoundT5 + 57k 17.0% top-1 exact, 23.7% top-5 exact, 34.0% top-5 core.

**Data:** `kuzmenkooleh/retro-planner-ord-150k-2` (147,000 train + 3,000 val, the same pool
variant 4 used). ~3 h 56 min training at the 0.97 steps/s measured on the 150k DDP run, plus
~15 min for the two evaluations.

**Before running:** Settings -> **Internet** on, **GPU T4 x2** on. Run as **Save & Run All
(Commit)**.

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available(), "| devices:", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(f"  Device {i}:", torch.cuda.get_device_name(i))

In [ ]:
import os
if not os.path.isdir("retro-planner"):
    !git clone https://github.com/oleh-kuzmenko/retro-planner.git
%cd retro-planner

In [ ]:
%pip install -q -e ".[local-models,indexing]"

In [ ]:
import glob

# Kaggle has mounted datasets under two different layouts historically, so search
# rather than hard-code the path.
train_file = next(glob.iglob("/kaggle/input/**/reactants_train.jsonl", recursive=True))
val_file = next(glob.iglob("/kaggle/input/**/reactants_val.jsonl", recursive=True))
for path in (train_file, val_file):
    print(path, sum(1 for _ in open(path)), "rows")

base_model = "sagawa/CompoundT5"
learning_rate = 5e-4  # same as the 57k run, so data volume is the only difference
output_dir = "/kaggle/working/model1_compoundt5_150k"
time_budget_minutes = 300  # ~236 min training + headroom

In [ ]:
import os

os.makedirs(output_dir, exist_ok=True)
log_path = f"{output_dir}/train.log"

!torchrun --nproc_per_node=2 scripts/train_reactant_model_ord.py \
    --base-model "{base_model}" \
    --train-file "{train_file}" \
    --val-file "{val_file}" \
    --output-dir "{output_dir}" \
    --local-work-dir /kaggle/temp/local_model1_work \
    --no-augment \
    --learning-rate {learning_rate} \
    --num-train-epochs 3 \
    --time-budget-minutes {time_budget_minutes} \
    > "{log_path}" 2>&1
print("training done; tail of log:")
!tail -5 "{log_path}"

In [ ]:
# The repair must have fired: 221 -> 251. If this line is absent the run trained on
# <unk>-corrupted targets and its numbers are meaningless.
!grep -E "new character token|Train examples" "{log_path}"

import json
from transformers import AutoTokenizer

saved_tokenizer = AutoTokenizer.from_pretrained(f"{output_dir}/final")
embedding_rows = json.load(open(f"{output_dir}/final/config.json"))["vocab_size"]
print("tokenizer length:", len(saved_tokenizer), "| embedding rows:", embedding_rows)
assert len(saved_tokenizer) == embedding_rows, "tokenizer and embedding matrix disagree"

In [ ]:
# Whether eval_loss was still falling at the end decides what the next run changes:
# still falling -> the budget binds, flattened well before the end -> the data does.
import json

state = json.load(open(f"{output_dir}/latest_checkpoint/trainer_state.json"))
points = [(h["epoch"], h["eval_loss"]) for h in state["log_history"] if "eval_loss" in h]
for epoch, loss in points[::max(1, len(points) // 10)]:
    print(f"  epoch {epoch:5.2f}  eval_loss {loss:.4f}")
print("  last:", points[-1], "| best:", state.get("best_metric"))

In [ ]:
import os
model_dir = f"{output_dir}/final"
assert os.path.isdir(model_dir), os.listdir(output_dir)

for tag, targets in [("ord", "data/v2_ord_eval_targets.json"),
                     ("uspto", "data/v2_uspto_eval_targets.json")]:
    !python scripts/models/run_reactiont5_topk.py \
        --input "{targets}" --t5-model "{model_dir}" \
        --num-beams 10 --device cuda \
        --output "/kaggle/working/compoundt5_150k_{tag}_topk.json"
    print(tag, "done")

In [ ]:
import json
for tag in ("ord", "uspto"):
    data = json.load(open(f"/kaggle/working/compoundt5_150k_{tag}_topk.json"))
    print("===", tag, "===")
    print(json.dumps(data["summary"], indent=2))